#OPT002 - Modelagem de problemas de otimização e implementação em AMPL




In [2]:
%pip install -q amplpy
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs", "cbc", "gurobi", "cplex", "gokestrel"], # pick from over 20 modules including most commercial and open-source solvers
    license_uuid="499a8852-4e10-4d0b-a0b7-e45ad6f0c1b2") # your license UUID (e.g., free ampl.com/ce or ampl.com/courses licenses)

Licensed to AMPL Community Edition License for <everton.santi@ufrn.br>.


# Modelagem de problemas de otimização

## Revisitando o Problema da Mochila (Knapsack Problem)

Suponha que você irá acampar no final de semana com alguns amigos. Procurando por seus materias de acampamento, acabou encontrando uma mochila com capacidade de carga de 10Kg.

Durante o processo de busca, encontrou também 6 itens que poderá levar nesta mochila. Uma consideração importante sobre estes itens é que ao somar o peso de cada um, o peso total ultrapassou a capacidade de carga da mochila. Logo, não será possível levar todos estes itens.

Para lhe auxiliar no processo de decisão de quais itens deverá levar, então, estabeleceu um sistema da pontuação, com notas de 1 a 10. Neste sistema, 10 significa que o item é crucial para o sucesso do acampamento, enquanto que 1 representa que o acampamento ocorreria sem maiores transtornos caso o item não seja levado. A lista dos itens disponíveis é a seguinte:

* Item 1: Lanterna (1Kg, 8 pontos)
* Item 2: Barraca (4Kg, 10 pontos)
* Item 3: Reservatório de água (3Kg, 10 pontos)
* Item 4: Reservatório de água (2Kg, 8 pontos)
* Item 5: Colchão Inflável (0.5Kg, 7 pontos)
* Item 6: JBL (2Kg, 4 pontos)

A solução do problema descrito consiste em decidir (**sim** ou **não**), se um item deverá ser levado na mochila considerando que queremos **maximizar** a soma das **pontuações** dos **itens escolhidos**. Ao mesmo tempo, a **combinação** de itens escolhidos **não** pode **ultrapassar** a **capacidade da mochila**.

### Formulação 1 (expandida)

$$
\text{maximize}~Z = 8 x_1 + 10 x_2 + 10 x_3 + 8 x_4 + 7 x_5 + 4 x_6  \tag{1}
$$
\
sujeito a:
\
$$
1 x_1 + 4 x_2 + 3 x_3 + 2 x_4 + 0.5 x_5 + 2 x_6 \leq 10 \tag{2}
$$
\
$$
x_1, x_2, x_3, x_4, x_5, x_6 \in \{0, 1\}  \tag{3}
$$
\
em que ($3$) maximiza a soma das pontuações dos itens escolhidos, (4) garante que a combinação escolhida não ultrapasse a capacidade de carga da mochila e (5) são as restrições de domínio sobre os valores das variáveis de decisão.
\

> ***Quais as limitações que podem ser observadas nesta maneira de formular o problema?***

### Formulação 2 (compacta)

O Problema da Mochila consiste em decidir quais, dentre um conjunto de $n$ itens, devem ser colocados na mochila, respeitando a sua capacidade de carga $C$, ao mesmo tempo em que maximizamos o benefício somado dos itens escolhidos.

De forma a definir o problema, consideram-se os seguintes parâmetros:

* $n$ - Total de itens disponíveis;
* $C$ - Capacidade de carga da mochila;
* $p_i$ - Peso do $i$-ésimo item, $∀ i=1, ..., n$;
* $b_i$ - Benefício do $i$-ésimo item, $∀ i=1, ..., n$;

Dado que solução do problema consiste em decidir se um item é colocado ou não na mochila, usa-se a variável de decisão $x_i$, $∀ i=1,...,n$, de tal modo que $x_i$ assume o valor 1 se o item $i$ é colocado na mochila, e zero caso contrário.

Considerando os parâmetros e variáveis de decisão listados, o Problema da Mochila poder ser formulado como:

$$
\begin{equation}
\textrm{maximize}~Z~=~\sum_{i=1}^{n}b_ix_i \tag{4}
\end{equation}
$$

Sujeito a:

$$
\begin{equation}
\sum_{i=1}^{n}p_ix_i \leq C \tag{5}
\end{equation}
$$

$$
x_i \in \{0,1\},~∀i=1,...,n \tag{6}
$$

em que (4) maximiza a soma dos benefícios dos itens colocados na mochila. A desigualdade em (5) garante que a capacidade de carga da mochila não será excedida e, por fim, as restrições em (6) são as restrições de domínio sobre as variáveis de decisão.



## Criando o arquivo .dat

In [3]:
%%writefile mochila.dat
param n := 6;
param C := 10;

#é possível criar uma tabela de parâmetros, desde que sejam relacionados ao mesmo
#conjunto de itens, neste caso os 6 itens a serem colocados na mochila
param : b   p  :=
        1   8  1
        2  10  4
        3  10  3
        4   8  2
        5   7  0.5
        6   4  2;

#é possível criar um parâmetro de forma isolada
#param p :=
#1 1
#2 4
#3 3
#4 2
#5 0.5
#6 2;


Writing mochila.dat


### Criando o arquivo .mod

In [4]:
%%writefile mochila.mod
param n > 0;
param C > 0;
param b{1..n};
param p{1..n};

var x{1..n} binary;

maximize Z: sum{i in 1..n}b[i]*x[i];

subject to capacidade: sum{i in 1..n}p[i]*x[i] <= C;


Writing mochila.mod


### Criando o arquivo .run

In [5]:
%%writefile mochila.run
model mochila.mod;
data mochila.dat;
#option solver highs;
#vamos utilizar aqui um serviço na nuvem,
#chamado neos server, que faz processamento
#oferendo solver comerciais sem custo,
#porém limitando o job a 8h e 3GB de memória
option solver kestrel;
option kestrel_options 'solver=cplex';
option cplex_options 'mipdisplay=2';
option email "santi.everton@gmail.com";
solve;
display Z;
display x;

Writing mochila.run


### Usando o terminal para executar

De posse dos três arquivos, podemos usar o terminal do sistema (Windows ou Linux) para executar nosso código.

In [6]:
%%shell
ampl mochila.run

Connecting to: neos-server.org:3333
Job 15866288 submitted to NEOS, password='zZgqFlYE'
Check the following URL for progress report:
https://neos-server.org/neos/cgi-bin/nph-neos-solver.cgi?admin=results&jobnumber=15866288&pass=zZgqFlYE
Job 15866288 dispatched
password: zZgqFlYE
---------- Begin Solver Output -----------
Condor submit: 'neos.submit'
Condor submit: 'watchdog.submit'
Job submitted to NEOS HTCondor pool.
kestrel_options:solver=cplex

cplex_options:mipdisplay=2

Supplied solver options are: mipdisplay=2 threads=4
You are using the solver cplex.
Executing on prod-exec-1.neos-server.org
CPLEX 22.1.2:   tech:mipdisplay = 2
  tech:threads = 4
CPLEX 22.1.2: optimal solution; objective 37
1 simplex iterations
Z = 37

x [*] :=
1  1
2  1
3  0
4  1
5  1
6  1
;



## Exercício em grupo

Para o problema descrito a seguir:

1.   Escreva uma formulação compacta, apresentando seu objetivo de modo geral, seus parâmetros e suas variáveis de decisão, bem como explicar cada uma de suas espressões ;
2.   Codifique em AMPL (.mod, .dat e .run) para obter a solução;

### Problema

> Os dados utilizados aqui foram retirados de [[Tirone, 2019]](https://imef.furg.br/images/stories/Monografias/Matematica_aplicada/2019/2019-2_Giulia.pdf)

Sua equipe está atuando no setor de planejamento do refeitório de uma fábrica, conjuntamente com a nutricionista responsável pelo estabelecimento. Diariamente, a empresa serve milhares de refeições aos trabalhadores, de tal modo que uma grande quantia é gasta todos os meses com este serviço.

No entando, há a necessidade de se **reduzir os custos** no preparo da refeições **sem prejudicar a saúde** dos trabalhadores. Logo, optou-se por criar um modelo de programação linear para auxiliar no processo de escolha dos alimentos que serão oferecidos aos trabalhadores em um dia específico, bem como sua respectiva quantidade.

A nutricionista estabeleceu quantidades mínimias para certos nutrientes, vitaminas e minerais que um adulto deve ingerir ao longo de um dia:

* Proteínas: 46g ou mais;
* Cálcio: 1g ou mais;
* Sódio: 1.5g ou mais;
* Ferro: 0.018g ou mais;
* Vitaminas: 0.771g ou mais;

A quantidade máxima de calorias ingeridas também foi fixada em 2000kcal.

Os alimentos disponíveis no estoque e suas respectivas informações são dados <a href="https://docs.google.com/spreadsheets/d/1bXXBPR11O2nOUxPc6lZZa7kLkg7ZWXAylVSi87lFGt0/edit?usp=sharing">aqui</a>.


## Escreva sua formulação aqui usando latex

Discuta com seu grupo:

* Quais são os parâmetros?
* Quais são as variáveis de decisão e qual seu tipo?
* Qual a função objetivo?
* Quais as restrições?

Apresente um modelo da mesma forma que o modelo (4-6) foi apresentado, explicando para que serve (significado) de cada um de seus elementos.

## Crie seu arquivo .dat

In [ ]:
%%writefile dieta.dat
param m := 13;
param n := 6;
param custo :=
             1 0.93
             2 5.24
             3 1.29
             4 3.05
             5 0.5
             6 0.63
             7 0.09
             8 0.89
             9 1.95
            10 0.21
            11 0.99
            12 0.61
            13 1.69 ;

param nome_item :=
             1 "pão integral"
             2 "queijo cottage"
             3 "mamão"
             4 "nozes"
             5 "salada crua"
             6 "feijão"
             7 "arroz integral"
             8 "frango grelhado"
             9 "maçã"
            10 "tapioca"
            11 "ovo"
            12 "atum ralado"
            13 "iogurte" ;

param :     nome_propriedade     l      u :=
        1   "calorias (kcal)"    0      2000
        2   "proteínas (g)"	     46     Infinity
        3   "cálcio (g)"         1      Infinity
        4   "sódio (g)"	         1.5    Infinity
        5   "ferro (g)"	         0.018  Infinity
        6   "vitaminas (g)"      0.77   Infinity ;
param p (tr):   1           2           3           4           5           6           7           8           9           10          11          12          13      :=
        1       136.000	    85.000	    45.000	    100.000	    25.000	    132.000	    25.000	    165.000	    72.000	    68.000	    77.000	    17.000	    99.000
        2         4.600	    17.200	     0.800	      2.300	     1.500	      8.800	     0.500	     31.000	     0.300	     0.000	     6.200	     3.000	     3.900
        3         0.060	     0.030	     0.020	      0.000	     0.020	      0.030	     0.000	      0.020	     0.010	     0.000	     0.030	     0.000	     0.140
        4         0.276	     0.013	     0.006	      0.000	     0.080	      0.001	     0.000	      0.074	     0.001	     0.037	     0.139	     0.069	     0.053
        5         0.100	     0.010	     0.010	      0.000	     0.030	      0.120	     0.000	      0.060	     0.010	     0.020	     0.030	     0.000	     0.000
        6         0.000	     0.010	     0.770	      0.000	     0.930	      0.000	     0.000	      0.000	     0.120	     0.000	     0.060	     0.000	     0.020;

Overwriting dieta.dat


## Crie seu arquivo .mod

In [ ]:
%%writefile dieta.mod
param m > 0;
param n > 0;
param custo{1..m} >= 0;
param nome_item{1..m} symbolic;
param nome_propriedade{1..n} symbolic;
param l{1..n} >=0;
param u{1..n}>=0;
param p{1..m, 1..n} >=0;

var x{1..m} >= 0;

minimize Z: sum{i in 1..m}custo[i]*x[i];

subject to r1{j in 1..n}:l[j] <= sum{i in 1..m}p[i,j]*x[i] <= u[j];

subject to r2: x[13] <= 1;

Overwriting dieta.mod


## Crie seu arquivo .run

In [ ]:
%%ampl_eval
reset;
model "dieta.mod";
data "dieta.dat";
option solver highs;
solve;
for{i in 1..m}
{
    if x[i] >=0 then
        printf "%s %f\n", nome_item[i], x[i];
}

HiGHS 1.7.0: HiGHS 1.7.0: optimal solution; objective 15.47163934
5 simplex iterations
0 barrier iterations
pão integral 13.540984
queijo cottage 0.000000
mamão 0.000000
nozes 0.000000
salada crua 2.377049
feijão 0.000000
arroz integral 0.000000
frango grelhado 0.000000
maçã 0.000000
tapioca 0.000000
ovo 0.000000
atum ralado 0.000000
iogurte 1.000000


In [ ]:
%%shell
ampl dieta.run

## Reflexões

1. Quais as limitações da solução apresentada para o problema?
2. O que poderia ser melhorado?
3. Em um cenário um pouco mais realista, como poderia ser pensado o cardápio de um refeitório como este descrito? Imagine inclusive o próprio restaurante universitário do campus, servindo milhares de refeições ao longo de uma semana, mês ou ano.
4. Este modelo poderia ser aplicado na sua casa ou trabalho?
5. Há semelhança com o problema da mochila?